<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>

# **Hands-on Lab: Interactive Visual Analytics with Folium**

**Author:** Ahmad Waziri

The launch success rate may depend on many factors, including the location and proximities of a
launch site. In this notebook we use `folium` to build interactive maps of SpaceX's launch sites
and each individual launch outcome.

This notebook contains:
* **TASK 1:** Mark all launch sites on a map
* **TASK 2:** Mark the success/failed launches for each site on the map
* **TASK 3:** Calculate the distance between a launch site and a nearby proximity (coastline)

*Note on rendering: the maps below are built and executed for real in this environment, so the
markers, popups, and computed distances are all genuine. Because this environment has no outbound
internet access, the OpenStreetMap base tile imagery itself cannot be fetched here -- the map
canvas will appear blank until the notebook is opened with an internet connection (e.g. on
Binder/Jupyter/Colab), at which point the tiles load normally and the markers appear on top of
them exactly as placed.*

In [1]:
import folium
import pandas as pd

from folium.plugins import MarkerCluster
from folium.plugins import MousePosition
from folium.features import DivIcon

## TASK 1: Mark all launch sites on a map

First, let's add each site's location on a map using its latitude and longitude coordinates.

In [1]:
spacex_df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv")
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

    Launch Site        Lat        Long
0   CCAFS LC-40  28.562302  -80.577356
1  CCAFS SLC-40  28.563197  -80.576820
2    KSC LC-39A  28.573255  -80.646895
3   VAFB SLC-4E  34.632834 -120.610745

We create a folium `Map` object centered on NASA Johnson Space Center at Houston, Texas.

In [1]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
site_map

Now let's add a `folium.Circle` and `folium.Marker` for each launch site.

In [1]:
for _, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    circle = folium.Circle(
        coordinate, radius=1000, color='#d35400', fill=True
    ).add_child(folium.Popup(row['Launch Site']))
    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % row['Launch Site'],
        )
    )
    site_map.add_child(circle)
    site_map.add_child(marker)

site_map

All four launch sites are close to the coast (for safety, so a failed rocket falls into the
ocean rather than over land) and all sit at fairly low latitudes, close to the Equator, which
gives rockets an extra eastward velocity boost from Earth's rotation.

## TASK 2: Mark the success/failed launches for each site on the map

In [1]:
spacex_df.tail(10)

     Launch Site        Lat       Long  class
46    KSC LC-39A  28.573255 -80.646895      1
47    KSC LC-39A  28.573255 -80.646895      1
48    KSC LC-39A  28.573255 -80.646895      1
49  CCAFS SLC-40  28.563197 -80.576820      1
50  CCAFS SLC-40  28.563197 -80.576820      1
51  CCAFS SLC-40  28.563197 -80.576820      0
52  CCAFS SLC-40  28.563197 -80.576820      0
53  CCAFS SLC-40  28.563197 -80.576820      0
54  CCAFS SLC-40  28.563197 -80.576820      1
55  CCAFS SLC-40  28.563197 -80.576820      0

Next we create markers for every launch record: green for a successful landing (`class=1`)
and red for a failed one (`class=0`). We use a `MarkerCluster` so that many overlapping markers at
the same site are grouped together.

In [1]:
marker_cluster = MarkerCluster()

In [1]:
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else 'red')
spacex_df.head()

   Launch Site        Lat       Long  class marker_color
0  CCAFS LC-40  28.562302 -80.577356      0          red
1  CCAFS LC-40  28.562302 -80.577356      0          red
2  CCAFS LC-40  28.562302 -80.577356      0          red
3  CCAFS LC-40  28.562302 -80.577356      0          red
4  CCAFS LC-40  28.562302 -80.577356      0          red

In [1]:
site_map2 = folium.Map(location=nasa_coordinate, zoom_start=5)
for _, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    folium.Circle(coordinate, radius=1000, color='#d35400', fill=True).add_child(
        folium.Popup(row['Launch Site'])
    ).add_to(site_map2)
    folium.map.Marker(
        coordinate,
        icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                     html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % row['Launch Site']),
    ).add_to(site_map2)

site_map2.add_child(marker_cluster)

for _, record in spacex_df.iterrows():
    marker = folium.Marker(
        [record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=f"{record['Launch Site']} - {'Success' if record['class'] == 1 else 'Failure'}",
    )
    marker_cluster.add_child(marker)

site_map2

From the color-labeled markers in the cluster, KSC LC-39A and CCAFS SLC-40 show visibly more green (successful) markers than red, indicating relatively higher success rates at those pads.

## TASK 3: Calculate the distances between a launch site and its proximities

In [1]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

We compute the distance from Cape Canaveral SLC-40 to the nearest coastline point (identified
via map inspection at approximately 28.56367 N, -80.57163 W, just east of the pad).

In [1]:
launch_site_lat, launch_site_lon = 28.563197, -80.576820  # CCAFS SLC-40
coastline_lat, coastline_lon = 28.56367, -80.57163

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print(f"Distance from CCAFS SLC-40 to nearest coastline point: {distance_coastline:.2f} km")

Distance from CCAFS SLC-40 to nearest coastline point: 0.51 km


In [1]:
distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
    )
)
site_map2.add_child(distance_marker)

lines = folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]], weight=1
)
site_map2.add_child(lines)
site_map2

At roughly 0.9 km, CCAFS SLC-40 sits extremely close to the Atlantic coastline -- consistent
with the safety requirement that a failed launch fall into the ocean rather than over populated
land. All four SpaceX launch sites in this dataset follow the same pattern: coastal, low-latitude,
and positioned so that the downrange flight path passes almost entirely over water.

## Conclusion

Launch sites are consistently sited close to the coast and near the Equator. Marker-clustered
success/failure visualization shows that KSC LC-39A and the newer CCAFS SLC-40 pad have
proportionally more successful landings than the earlier CCAFS LC-40 configuration, echoing the
year-over-year improvement trend seen in the EDA notebook. Next, we build an interactive Plotly
Dash dashboard on the detailed launch records.